# Week 4 Exercises: APIs, Data Wrangling & PostgreSQL

**Data Engineering Course · Pipeline Africa**

---

Work through these exercises in order. Each section builds on the last.

**Setup:**
```
pip install requests pandas psycopg2-binary python-dotenv
```

In [ ]:
import os
import requests
import pandas as pd
import json
from pathlib import Path

DATA = Path('../data')
print('Ready.')

---

## Section 1: REST APIs (Exercises 1–5)

### Exercise 1

Make a GET request to `https://api.open-meteo.com/v1/forecast` for Nairobi, Kenya (latitude `-1.2921`, longitude `36.8219`, timezone `Africa/Nairobi`).

Request these daily fields: `temperature_2m_max`, `temperature_2m_min`, `precipitation_sum`.

Print:
- The HTTP status code
- The timezone in the response
- The first date in the `daily.time` list

In [ ]:
# Exercise 1 — your code here


### Exercise 2

Write a function called `get_forecast(lat, lon, tz)` that:
- Makes a GET request to Open-Meteo
- Returns the full parsed JSON dict on success
- Returns `None` if the request fails (use `try/except`)
- Passes `timeout=10` to `requests.get()`

Test it with Accra, Ghana (lat `5.6037`, lon `-0.1870`, tz `Africa/Accra`).

In [ ]:
# Exercise 2 — your code here


### Exercise 3

What happens when you send a bad status code? Call the URL below and:
- Print the status code
- Print the JSON error message from the response

```python
bad_url = 'https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&timezone=UTC'
```

In [ ]:
# Exercise 3 — your code here
bad_url = 'https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&timezone=UTC'


### Exercise 4

Fetch the forecast for Cairo (lat `30.0444`, lon `31.2357`, tz `Africa/Cairo`) and save the raw JSON response to `../data/cairo_forecast.json`.

Then read the file back and print the number of days in the response.

In [ ]:
# Exercise 4 — your code here


### Exercise 5

The Open-Meteo API supports hourly data as well as daily. Make a request for Lagos (lat `6.5244`, lon `3.3792`) that asks for the **hourly** field `temperature_2m`. You will need to change the `daily` parameter to `hourly`.

Print how many hourly values are returned. (Hint: 7 days × 24 hours = ?)

In [ ]:
# Exercise 5 — your code here


---

## Section 2: Data Wrangling (Exercises 6–10)

### Exercise 6

Load `../data/sample_weather.json` and build a DataFrame from the `daily` data.

Rename the columns:
- `time` → `date`
- `temperature_2m_max` → `temp_max_c`
- `temperature_2m_min` → `temp_min_c`
- `precipitation_sum` → `rain_mm`
- `windspeed_10m_max` → `wind_max_kmh`

Print the DataFrame.

In [ ]:
# Exercise 6 — your code here


### Exercise 7

Add a column called `temp_range_c` that shows the difference between `temp_max_c` and `temp_min_c` for each day.

Add a column called `is_rainy` that is `True` when `rain_mm > 0`, and `False` otherwise.

Print the updated DataFrame.

In [ ]:
# Exercise 7 — your code here


### Exercise 8

Fetch the 7-day forecast for all 5 cities in `../data/cities.csv` using the `get_forecast()` function from Exercise 2.

Build a combined DataFrame. It must include a `city` column so you can tell the rows apart.

Print the shape of the combined DataFrame and the first 5 rows.

In [ ]:
# Exercise 8 — your code here
import time


### Exercise 9

Using the combined DataFrame from Exercise 8:

1. Which city has the highest **average maximum temperature** this week?
2. Which city has the highest **total rainfall** this week?
3. On how many days does Cairo have wind speed above 20 km/h?

Answer each question with a short pandas expression.

In [ ]:
# Exercise 9 — your code here


### Exercise 10

Write a validation function called `check_weather_data(df)` that:
- Checks there are no null values in `temp_max_c`, `temp_min_c`
- Checks that `temp_max_c > temp_min_c` for every row
- Checks that `rain_mm >= 0` for every row
- Prints 'Validation passed' if everything is OK
- Prints a specific error message for each problem it finds

Run it on your combined DataFrame.

In [ ]:
# Exercise 10 — your code here


---

## Section 3: PostgreSQL (Exercises 11–15)

*These exercises require PostgreSQL to be running.*

### Exercise 11

Connect to PostgreSQL using credentials from environment variables (use `os.environ.get()` with sensible defaults).

Print the server version to confirm the connection works.

In [ ]:
# Exercise 11 — your code here
import psycopg2
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


### Exercise 12

Create a table called `city_weather_ex` with these columns:
- `id` SERIAL PRIMARY KEY
- `city` VARCHAR(100) NOT NULL
- `date` DATE NOT NULL
- `temp_max_c` NUMERIC(5,2)
- `rain_mm` NUMERIC(7,2)
- UNIQUE constraint on (city, date)

Use `CREATE TABLE IF NOT EXISTS` so it is safe to re-run.

In [ ]:
# Exercise 12 — your code here


### Exercise 13

Insert the rows from your combined DataFrame (Exercise 8) into `city_weather_ex`.

Use `cur.executemany()` with a parameterised `INSERT` statement.

After inserting, print the total row count from the database.

In [ ]:
# Exercise 13 — your code here


### Exercise 14

Use `pd.read_sql()` to answer this question from the database:

*Which city had the highest maximum temperature recorded in the table?*

Write a SQL query that returns the city name, date, and `temp_max_c`, ordered by temperature descending, limited to 1 row.

In [ ]:
# Exercise 14 — your code here


### Exercise 15

Change your INSERT from Exercise 13 to an **upsert** (`INSERT ... ON CONFLICT DO UPDATE`).

Run it twice. Confirm that the row count stays the same after the second run — no duplicates.

In [ ]:
# Exercise 15 — your code here


---

## Section 4: Full Pipeline Challenge (Exercises 16–18)

### Exercise 16

Add logging to the `get_forecast()` function from Exercise 2.

Configure a logger that writes to `../logs/exercises.log` AND the console.

Log:
- `INFO` when a city fetch succeeds (include the city name and number of rows)
- `ERROR` when a fetch fails (include the city name and the error)

In [ ]:
# Exercise 16 — your code here
import logging


### Exercise 17

Write a `config.json` file for your pipeline. It should contain:
- `api.base_url`
- `api.timeout_sec`
- `api.fields`
- `cities_file`
- `output_dir`

Write a function `load_config(path)` that reads the JSON file and returns a dict. Test it by printing the API base URL.

In [ ]:
# Exercise 17 — your code here


### Exercise 18 — Full Pipeline

Build a single function `run_pipeline(config_path)` that:

1. Loads the config from `config.json`
2. Loads cities from the cities CSV
3. Fetches the 7-day forecast for each city (with a 0.3-second pause between requests)
4. Validates the data
5. Saves to `data/clean/pipeline_output.csv`
6. Logs every major step

Run it and verify the output CSV exists and has the right shape.

In [ ]:
# Exercise 18 — your code here


---

<details>
<summary><strong>Solutions (click to expand)</strong></summary>

### Exercise 1 solution
```python
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude':  -1.2921,
    'longitude': 36.8219,
    'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone':  'Africa/Nairobi',
}
response = requests.get(url, params=params)
data = response.json()
print(f'Status: {response.status_code}')
print(f'Timezone: {data["timezone"]}')
print(f'First date: {data["daily"]["time"][0]}')
```

### Exercise 2 solution
```python
def get_forecast(lat, lon, tz, fields='temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'):
    params = {
        'latitude':  lat,
        'longitude': lon,
        'daily':     fields,
        'timezone':  tz,
    }
    try:
        response = requests.get(
            'https://api.open-meteo.com/v1/forecast',
            params=params,
            timeout=10,
        )
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f'Error: {e}')
        return None

result = get_forecast(5.6037, -0.1870, 'Africa/Accra')
if result:
    print('Got', len(result['daily']['time']), 'days for Accra')
```

### Exercise 3 solution
```python
response = requests.get(bad_url)
print(f'Status: {response.status_code}')
print(response.json())
```

### Exercise 4 solution
```python
data = get_forecast(30.0444, 31.2357, 'Africa/Cairo')
if data:
    out = DATA / 'cairo_forecast.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
    with open(out, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    print(f'Days: {len(loaded["daily"]["time"])}')
```

### Exercise 5 solution
```python
params = {
    'latitude':  6.5244,
    'longitude': 3.3792,
    'hourly':    'temperature_2m',
    'timezone':  'Africa/Lagos',
}
response = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
data = response.json()
print(f'Hourly values: {len(data["hourly"]["time"])}')  # 168
```

### Exercise 6 solution
```python
with open(DATA / 'sample_weather.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)
df = pd.DataFrame(raw['daily']).rename(columns={
    'time':                'date',
    'temperature_2m_max':  'temp_max_c',
    'temperature_2m_min':  'temp_min_c',
    'precipitation_sum':   'rain_mm',
    'windspeed_10m_max':   'wind_max_kmh',
})
print(df)
```

### Exercise 7 solution
```python
df['temp_range_c'] = df['temp_max_c'] - df['temp_min_c']
df['is_rainy']     = df['rain_mm'] > 0
print(df)
```

### Exercise 8 solution
```python
cities_df = pd.read_csv(DATA / 'cities.csv')
all_frames = []
for _, row in cities_df.iterrows():
    raw = get_forecast(row['latitude'], row['longitude'], row['timezone'])
    if raw:
        df = pd.DataFrame(raw['daily']).rename(columns={
            'time': 'date', 'temperature_2m_max': 'temp_max_c',
            'temperature_2m_min': 'temp_min_c', 'precipitation_sum': 'rain_mm',
            'windspeed_10m_max': 'wind_max_kmh',
        })
        df['city'] = row['city']
        all_frames.append(df)
    time.sleep(0.3)
combined = pd.concat(all_frames, ignore_index=True)
print(combined.shape)
print(combined.head())
```

### Exercise 9 solution
```python
# 1
print(combined.groupby('city')['temp_max_c'].mean().idxmax())
# 2
print(combined.groupby('city')['rain_mm'].sum().idxmax())
# 3
cairo = combined[combined['city'] == 'Cairo']
print((cairo['wind_max_kmh'] > 20).sum())
```

### Exercise 10 solution
```python
def check_weather_data(df):
    ok = True
    if df['temp_max_c'].isnull().any():
        print('ERROR: temp_max_c has null values'); ok = False
    if df['temp_min_c'].isnull().any():
        print('ERROR: temp_min_c has null values'); ok = False
    if not (df['temp_max_c'] > df['temp_min_c']).all():
        print('ERROR: temp_max_c is not always greater than temp_min_c'); ok = False
    if (df['rain_mm'] < 0).any():
        print('ERROR: rain_mm has negative values'); ok = False
    if ok:
        print('Validation passed')
check_weather_data(combined)
```

### Exercise 15 key — upsert pattern
```python
upsert_sql = '''
    INSERT INTO city_weather_ex (city, date, temp_max_c, rain_mm)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (city, date) DO UPDATE SET
        temp_max_c = EXCLUDED.temp_max_c,
        rain_mm    = EXCLUDED.rain_mm;
'''
```

</details>